In [1]:
import pandas as pd
import numpy as np
from scipy import stats

# 1. Cargar dataset
ruta = r"C:\Users\Martín\Desktop\POLARIZACOIN\resultados\dataset_comuna_completo.csv"
df = pd.read_csv(ruta)

# 2. Agrupar a nivel de distrito (N=28)
# La polarización es única por distrito, las demás variables se promedian
vars_independientes = ['promedio_escolaridad', 'tasa_pobreza_ingresos_2024', 'pct_migrantes', 
                       'edad_promedio', 'tasa_delitos_100k', 'pct_evangelicos_limpio', 
                       'pct_sin_religion_limpio', 'pct_rural']

# Creamos el dataframe a nivel distrital
df_distrito = df.groupby('Distrito')[vars_independientes + ['polarizacion_distrito']].mean().reset_index()

print(f"Observaciones originales (comunas): {len(df)}")
print(f"Observaciones corregidas (distritos): {len(df_distrito)}")

# 3. Regresión a nivel distrital
X = df_distrito[vars_independientes]
y = df_distrito['polarizacion_distrito']

# Estandarizar
X_z = (X - X.mean()) / X.std()
y_z = (y - y.mean()) / y.std()

# Añadir intercepto
X_z = np.column_stack([np.ones(len(X_z))] + [X_z[col].values for col in vars_independientes])

# Calcular coeficientes (OLS)
coefs = np.linalg.lstsq(X_z, y_z, rcond=None)[0]
residuos = y_z - (X_z @ coefs)
r2 = 1 - (np.sum(residuos ** 2) / np.sum((y_z - y_z.mean()) ** 2))

# Errores estándar y p-valores
n = len(y_z)
k = X_z.shape[1] - 1
se = np.sqrt(np.diag((np.sum(residuos ** 2) / (n - k - 1)) * np.linalg.inv(X_z.T @ X_z)))
t_stats = coefs / se
p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), df=n - k - 1))

print(f"\n=== REGRESIÓN NIVEL DISTRITAL (N=28) ===")
print(f"R² = {r2:.3f}")
print(f"{'Variable':<30} {'Coef':>8} {'p-valor':>10}")
print("-" * 55)
for i, var in enumerate(vars_independientes, 1):
    sig = "***" if p_values[i]<0.001 else "**" if p_values[i]<0.01 else "*" if p_values[i]<0.05 else ""
    print(f"{var:<30} {coefs[i]:>8.3f} {p_values[i]:>10.4f} {sig}")

print("\nNOTA: Comparar estos p-valores con los del modelo original (N=346).")

Observaciones originales (comunas): 346
Observaciones corregidas (distritos): 28

=== REGRESIÓN NIVEL DISTRITAL (N=28) ===
R² = 0.388
Variable                           Coef    p-valor
-------------------------------------------------------
promedio_escolaridad              0.738     0.1485 
tasa_pobreza_ingresos_2024        0.406     0.3768 
pct_migrantes                    -0.183     0.5630 
edad_promedio                    -0.426     0.2819 
tasa_delitos_100k                -0.152     0.5965 
pct_evangelicos_limpio           -0.084     0.7496 
pct_sin_religion_limpio          -0.358     0.2808 
pct_rural                        -0.236     0.4532 

NOTA: Comparar estos p-valores con los del modelo original (N=346).


In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler

# 1. Cargar y agrupar a nivel distrital
ruta = r"C:\Users\Martín\Desktop\POLARIZACOIN\resultados\dataset_comuna_completo.csv"
df = pd.read_csv(ruta)

vars_independientes = ['promedio_escolaridad', 'tasa_pobreza_ingresos_2024', 'pct_migrantes', 
                       'edad_promedio', 'tasa_delitos_100k', 'pct_evangelicos_limpio', 
                       'pct_sin_religion_limpio', 'pct_rural']

df_distrito = df.groupby('Distrito')[vars_independientes + ['polarizacion_distrito']].mean().reset_index()

X = df_distrito[vars_independientes]
y = df_distrito['polarizacion_distrito']

# 2. Estandarizar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Modelo Lasso con validación cruzada (Leave-One-Out por N pequeño)
# alpha controla la penalización. LassoCV busca el mejor alpha automáticamente.
lasso = LassoCV(cv=len(X), random_state=42, max_iter=10000).fit(X_scaled, y)

print("=== MODELO LASSO (Nivel Distrital, N=28) ===")
print(f"Mejor alpha (penalización): {lasso.alpha_:.4f}")
print(f"R² del modelo: {lasso.score(X_scaled, y):.3f}")
print("\nCoeficientes (las variables en 0 fueron eliminadas por el modelo):")
print("-" * 55)

for var, coef in zip(vars_independientes, lasso.coef_):
    print(f"{var:<30} {coef:>8.3f}")

# 4. Baseline trivial (predecir la media)
baseline_pred = np.full(len(y), y.mean())
baseline_mse = np.mean((y - baseline_pred) ** 2)
modelo_mse = np.mean((y - lasso.predict(X_scaled)) ** 2)

print(f"\nMSE Baseline (predecir la media): {baseline_mse:.4f}")
print(f"MSE Modelo Lasso: {modelo_mse:.4f}")

=== MODELO LASSO (Nivel Distrital, N=28) ===
Mejor alpha (penalización): 0.0479
R² del modelo: 0.296

Coeficientes (las variables en 0 fueron eliminadas por el modelo):
-------------------------------------------------------
promedio_escolaridad              0.040
tasa_pobreza_ingresos_2024       -0.000
pct_migrantes                     0.000
edad_promedio                    -0.000
tasa_delitos_100k                -0.000
pct_evangelicos_limpio           -0.000
pct_sin_religion_limpio          -0.000
pct_rural                        -0.104

MSE Baseline (predecir la media): 0.1085
MSE Modelo Lasso: 0.0764


In [3]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

# 1. Cargar dataset
ruta = r"C:\Users\Martín\Desktop\POLARIZACOIN\resultados\dataset_comuna_completo.csv"
df = pd.read_csv(ruta)

# 2. Calcular tasa de participación (Votos totales / Población 18+)
# Votos TRICEL en el dataset comunal representa el total de votos válidos de la comuna
df['tasa_participacion'] = df['Votos TRICEL'] / df['poblacion_18mas']

print("=== ESTADÍSTICAS DE PARTICIPACIÓN (2021) ===")
print(df['tasa_participacion'].describe().round(3).to_string())

# 3. Correlación entre participación y educación
subset = df[['tasa_participacion', 'promedio_escolaridad', 'pct_rural']].dropna()
r_edu, p_edu = spearmanr(subset['tasa_participacion'], subset['promedio_escolaridad'])
r_rur, p_rur = spearmanr(subset['tasa_participacion'], subset['pct_rural'])

print(f"\nCorrelación Participación vs Escolaridad: r = {r_edu:.3f}, p = {p_edu:.4f}")
print(f"Correlación Participación vs Ruralidad:   r = {r_rur:.3f}, p = {p_rur:.4f}")

# 4. Agregar al nivel distrital y correr regresión OLS
vars_independientes = ['promedio_escolaridad', 'pct_rural', 'tasa_participacion']
df_distrito = df.groupby('Distrito')[vars_independientes + ['polarizacion_distrito']].mean().reset_index()

X = df_distrito[vars_independientes]
y = df_distrito['polarizacion_distrito']

# Estandarizar
X_z = (X - X.mean()) / X.std()
y_z = (y - y.mean()) / y.std()
X_z = np.column_stack([np.ones(len(X_z))] + [X_z[col].values for col in vars_independientes])

# OLS
coefs = np.linalg.lstsq(X_z, y_z, rcond=None)[0]
residuos = y_z - (X_z @ coefs)
r2 = 1 - (np.sum(residuos ** 2) / np.sum((y_z - y_z.mean()) ** 2))

n = len(y_z)
k = X_z.shape[1] - 1
se = np.sqrt(np.diag((np.sum(residuos ** 2) / (n - k - 1)) * np.linalg.inv(X_z.T @ X_z)))
t_stats = coefs / se
p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), df=n - k - 1))

print(f"\n=== NUEVO MODELO CON PARTICIPACIÓN (N=28) ===")
print(f"R² = {r2:.3f}")
print(f"{'Variable':<30} {'Coef':>8} {'p-valor':>10}")
print("-" * 55)
for i, var in enumerate(vars_independientes, 1):
    sig = "***" if p_values[i]<0.001 else "**" if p_values[i]<0.01 else "*" if p_values[i]<0.05 else ""
    print(f"{var:<30} {coefs[i]:>8.3f} {p_values[i]:>10.4f} {sig}")

=== ESTADÍSTICAS DE PARTICIPACIÓN (2021) ===
count    346.000
mean       0.471
std        0.233
min        0.223
25%        0.391
50%        0.430
75%        0.477
max        3.056

Correlación Participación vs Escolaridad: r = 0.264, p = 0.0000
Correlación Participación vs Ruralidad:   r = -0.075, p = 0.1649

=== NUEVO MODELO CON PARTICIPACIÓN (N=28) ===
R² = 0.342
Variable                           Coef    p-valor
-------------------------------------------------------
promedio_escolaridad              0.329     0.2407 
pct_rural                        -0.264     0.3565 
tasa_participacion               -0.178     0.3797 


In [4]:
import pandas as pd

ruta = r"C:\Users\Martín\Desktop\POLARIZACOIN\resultados\dataset_comuna_completo.csv"
df = pd.read_csv(ruta)

# Recalcular tasa de participación
df['tasa_participacion'] = df['Votos TRICEL'] / df['poblacion_18mas']

# Filtrar anomalías (participación > 100%)
anomalias = df[df['tasa_participacion'] > 1.0]

print("=== COMUNAS CON PARTICIPACIÓN > 100% ===")
columnas_mostrar = ['Comuna', 'cod_com', 'Votos TRICEL', 'poblacion_18mas', 'tasa_participacion']
print(anomalias[columnas_mostrar].sort_values('tasa_participacion', ascending=False).to_string(index=False))

=== COMUNAS CON PARTICIPACIÓN > 100% ===
          Comuna  cod_com  Votos TRICEL  poblacion_18mas  tasa_participacion
TORRES DEL PAINE    12402         550.0              180            3.055556
       RIO VERDE    12103         233.0               96            2.427083
         OLLAGUE     2202         329.0              208            1.581731
           PUTRE    15201        1871.0             1297            1.442560
        COLCHANE     1403         811.0              596            1.360738
       CAMARONES    15102        1025.0              755            1.357616
   GENERAL LAGOS    15202         474.0              385            1.231169
   LAGUNA BLANCA    12102         285.0              248            1.149194
        TIMAUKEL    12303         145.0              130            1.115385
    SAN GREGORIO    12104         223.0              208            1.072115
